# Food Hazard Detection ST1 Challenge
##TF-IDF Hazard + TF-IDF/MiniLM Product model

ELENI PAPAZOGLOU 5323 \
AGLAIA LIANOU 5268



## 1. Setup and imports

In [ ]:
!pip install -q sentence-transformers

In [ ]:
import os
import pandas as pd
import numpy as np
import torch

from scipy.sparse import hstack, csr_matrix

from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

from sentence_transformers import SentenceTransformer

## 2. Load the data

In [ ]:
# To run this notebook, create a folder in your Google Drive named: NLP_Food_Hazard_project
# The folder must contain the following files: train.csv, valid.csv, test.csv
# If your files are stored in another folder, change "DATA_DIR" accordingly.

from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/NLP_Food_Hazard_project"

print(os.listdir(DATA_DIR))

Mounted at /content/drive
['test.csv', 'train.csv', 'valid.csv', 'food_hazard_detection.ipynb', 'roberta_hazard_model', 'embeddings_mpnet', 'submission_champion_tfidf_ensemble.csv']


In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv", encoding="utf-8")
valid = pd.read_csv(f"{DATA_DIR}/valid.csv", encoding="utf-8")
test = pd.read_csv(f"{DATA_DIR}/test.csv", encoding="utf-8")

print("Train shape:", train.shape)
print("Valid shape:", valid.shape)
print("Test shape:", test.shape)

train.head()

Train shape: (5082, 11)
Valid shape: (565, 11)
Test shape: (997, 7)


,Unnamed: 0,year,month,day,country,title,text,hazard-category,product-category,hazard,product
0,0,1994,1,7,us,Recall Notification: FSIS-024-94,Case Number: 024-94 \n Date Opene...,biological,"meat, egg and dairy products",listeria monocytogenes,smoked sausage
1,1,1994,3,10,us,Recall Notification: FSIS-033-94,Case Number: 033-94 \n Date Opene...,biological,"meat, egg and dairy products",listeria spp,sausage
2,2,1994,3,28,us,Recall Notification: FSIS-014-94,Case Number: 014-94 \n Date Opene...,biological,"meat, egg and dairy products",listeria monocytogenes,ham slices
3,3,1994,4,3,us,Recall Notification: FSIS-009-94,Case Number: 009-94 \n Date Opene...,foreign bodies,"meat, egg and dairy products",plastic fragment,thermal processed pork meat
4,4,1994,7,1,us,Recall Notification: FSIS-001-94,Case Number: 001-94 \n Date Opene...,foreign bodies,"meat, egg and dairy products",plastic fragment,chicken breast


In [ ]:
# Basic dataset checks.

print("Missing values in train:")
print(train.isnull().sum())

print("\nMissing values in valid:")
print(valid.isnull().sum())

print("\nMissing values in test:")
print(test.isnull().sum())

print("\nNumber of categories:")
print("Hazard categories in train:", train["hazard-category"].nunique())
print("Product categories in train:", train["product-category"].nunique())
print("Hazard categories in valid:", valid["hazard-category"].nunique())
print("Product categories in valid:", valid["product-category"].nunique())

Missing values in train:
Unnamed: 0          0
year                0
month               0
day                 0
country             0
title               0
text                0
hazard-category     0
product-category    0
hazard              0
product             0
dtype: int64

Missing values in valid:
Unnamed: 0          0
year                0
month               0
day                 0
country             0
title               0
text                0
hazard-category     0
product-category    0
hazard              0
product             0
dtype: int64

Missing values in test:
id         0
year       0
month      0
day        0
country    0
title      0
text       0
dtype: int64

Number of categories:
Hazard categories in train: 10
Product categories in train: 22
Hazard categories in valid: 9
Product categories in valid: 18


## 3. Helper functions



In [ ]:
def make_input_text(df, title_repeat=2, use_meta=False):
    """Create the text representation used as model input."""

    title = df["title"].fillna("").astype(str)
    text = df["text"].fillna("").astype(str)

    #Repeat the title according to title_repeat.
    #We use different title repetitions for the hazard model and the product model.
    title_part = title.copy()
    for _ in range(title_repeat - 1):
        title_part = title_part + " " + title

    final_text = title_part + " " + text

    #When use_meta = True
    if use_meta:
        if "country" in df.columns:
            final_text = final_text + " country_" + df["country"].fillna("").astype(str)
        if "year" in df.columns:
            final_text = final_text + " year_" + df["year"].fillna("").astype(str)
        if "month" in df.columns:
            final_text = final_text + " month_" + df["month"].fillna("").astype(str)

    return final_text

In [ ]:
#Compute the official validation score
def official_score(y_true_hazard, y_pred_hazard, y_true_product, y_pred_product):

    f1_hazard = f1_score(
        y_true_hazard,
        y_pred_hazard,
        average="macro",
        zero_division=0
    )

    mask = np.array(y_true_hazard) == np.array(y_pred_hazard)

    f1_product_when_hazard_correct = f1_score(
        np.array(y_true_product)[mask],
        np.array(y_pred_product)[mask],
        average="macro",
        zero_division=0
    )

    score = (f1_hazard + f1_product_when_hazard_correct) / 2

    return f1_hazard, f1_product_when_hazard_correct, score

In [ ]:
# Target labels.

y_train_hazard = train["hazard-category"]
y_valid_hazard = valid["hazard-category"]

y_train_product = train["product-category"]
y_valid_product = valid["product-category"]

## 4. Hazard-category model: TF-IDF + SGDClassifier

The input text uses:
- title repeated 2 times
- full recall text
- metadata tokens for country, year, and month

The TF-IDF representation combines:
- word n-grams
- character n-grams

The classifier is SGDClassifier with hinge loss.


In [ ]:
# Create the input text for the hazard model.
# We repeat the title two times and also add metadata.
X_train_hazard = make_input_text(train, title_repeat=2, use_meta=True)
X_valid_hazard = make_input_text(valid, title_repeat=2, use_meta=True)
X_test_hazard = make_input_text(test, title_repeat=2, use_meta=True)

# Word-level TF-IDF features.
# These capture important words and word pairs
word_tfidf_hazard = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    max_features=70000,
    sublinear_tf=True,
    min_df=2,
    strip_accents="unicode"
)

# Character-level TF-IDF features.
# These capture small character patterns and help with word variations.
char_tfidf_hazard = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 6),
    max_features=100000,
    sublinear_tf=True,
    min_df=2,
    strip_accents="unicode"
)

# Combine word-level and character-level TF-IDF features.
tfidf_hazard = FeatureUnion([
    ("word", word_tfidf_hazard),
    ("char", char_tfidf_hazard)
])

# Fit the TF-IDF representation on the training data.
# Then transform validation and test using the same learned vocabulary.
X_train_hazard_tfidf = tfidf_hazard.fit_transform(X_train_hazard)
X_valid_hazard_tfidf = tfidf_hazard.transform(X_valid_hazard)
X_test_hazard_tfidf = tfidf_hazard.transform(X_test_hazard)

# Define the hazard classifier.
# With loss="hinge", SGDClassifier works like a linear SVM.
hazard_clf = SGDClassifier(
    loss="hinge",
    alpha=4e-5,
    penalty="l2",
    class_weight="balanced",
    max_iter=3000,
    tol=1e-4,
    random_state=42
)

# Train the hazard model on the TF-IDF training features.
hazard_clf.fit(X_train_hazard_tfidf, y_train_hazard)

# Predict hazard labels for validation and test data.
pred_valid_hazard = hazard_clf.predict(X_valid_hazard_tfidf)
pred_test_hazard = hazard_clf.predict(X_test_hazard_tfidf)

# Compute macro F1-score on the validation set.
hazard_f1 = f1_score(
    y_valid_hazard,
    pred_valid_hazard,
    average="macro",
    zero_division=0
)

print("Validation F1 hazard:", hazard_f1)

Validation F1 hazard: 0.8582462820456271


## 5. Product-category TF-IDF features

The input text uses:
- title repeated 3 times
- full recall text
- metadata tokens for country, year, and month

We do not train the final product classifier here yet.  
These TF-IDF features will be combined with MiniLM embeddings in the next steps.


In [ ]:
# Create the input text for the product model.
# For product, we repeat the title 3 times because the product name
# is often clearly mentioned in the title.
X_train_product = make_input_text(train, title_repeat=3, use_meta=True)
X_valid_product = make_input_text(valid, title_repeat=3, use_meta=True)
X_test_product = make_input_text(test, title_repeat=3, use_meta=True)

# Word-level TF-IDF features for product classification.
# We use up to trigrams because product categories often appear as longer phrases.
word_tfidf_product = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 3),
    max_features=70000,
    sublinear_tf=True,
    min_df=2,
    strip_accents="unicode"
)

# Character-level TF-IDF features for product classification.
# These help with product name variations and spelling differences.
char_tfidf_product = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    max_features=50000,
    sublinear_tf=True,
    min_df=2,
    strip_accents="unicode"
)

# Combine word-level and character-level TF-IDF features.
tfidf_product = FeatureUnion([
    ("word", word_tfidf_product),
    ("char", char_tfidf_product)
])

# Fit the TF-IDF representation on the training data.
# Then transform validation and test using the same learned vocabulary.
X_train_product_tfidf = tfidf_product.fit_transform(X_train_product)
X_valid_product_tfidf = tfidf_product.transform(X_valid_product)
X_test_product_tfidf = tfidf_product.transform(X_test_product)


print("Product TF-IDF train shape:", X_train_product_tfidf.shape)
print("Product TF-IDF valid shape:", X_valid_product_tfidf.shape)
print("Product TF-IDF test shape:", X_test_product_tfidf.shape)

Product TF-IDF train shape: (5082, 120000)
Product TF-IDF valid shape: (565, 120000)
Product TF-IDF test shape: (997, 120000)


## 6. Product-category MiniLM embeddings

We use all-MiniLM-L6-v2 to create dense 384-dimensional sentence embeddings for the product model.




In [ ]:
# Use GPU if it is available, otherwise use CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"


# Load the all-MiniLM-L6-v2 SentenceTransformer model.
# This model converts each recall text into a dense embedding vector.
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)


# Create the text input for the MiniLM product embeddings.
# We repeat the title 3 times, but we do not add metadata here.
X_train_emb_product_text = make_input_text(train, title_repeat=3, use_meta=False)
X_valid_emb_product_text = make_input_text(valid, title_repeat=3, use_meta=False)
X_test_emb_product_text = make_input_text(test, title_repeat=3, use_meta=False)


print("Creating product embeddings...")


# Encode training texts into normalized dense vectors.
X_train_emb_product = embedding_model.encode(
    X_train_emb_product_text.tolist(),
    batch_size=32,              # Number of texts processed at the same time
    show_progress_bar=True,     # Show encoding progress
    normalize_embeddings=True   # Normalize vectors for more stable classification
)


# Encode validation texts using the same embedding model.
X_valid_emb_product = embedding_model.encode(
    X_valid_emb_product_text.tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)


# Encode test texts using the same embedding model.
X_test_emb_product = embedding_model.encode(
    X_test_emb_product_text.tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)


# Print the embedding matrix shapes.
# Each text is represented by a 384-dimensional MiniLM vector.
print("Product embeddings:")
print("Train:", X_train_emb_product.shape)
print("Valid:", X_valid_emb_product.shape)
print("Test:", X_test_emb_product.shape)

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

Creating product embeddings...


Batches:   0%|          | 0/159 [00:00<?, ?it/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Product embeddings:
Train: (5082, 384)
Valid: (565, 384)
Test: (997, 384)


## 7. Final product-category model: TF-IDF + MiniLM

Using a small validation script, we found that the best values were:
- embedding_scale = 0.5
- C = 2.0




In [ ]:
# Best parameters selected from previous experiments.
FINAL_EMBEDDING_SCALE = 0.5
FINAL_PRODUCT_C = 2.0         # Regularization parameter for LinearSVC


# Combine TF-IDF product features with MiniLM embeddings for the training set.
X_train_product_final = hstack([
    X_train_product_tfidf,
    csr_matrix(X_train_emb_product * FINAL_EMBEDDING_SCALE)
])


# Combine TF-IDF features and MiniLM embeddings for the validation set.
X_valid_product_final = hstack([
    X_valid_product_tfidf,
    csr_matrix(X_valid_emb_product * FINAL_EMBEDDING_SCALE)
])


# Combine TF-IDF features and MiniLM embeddings for the test set.
X_test_product_final = hstack([
    X_test_product_tfidf,
    csr_matrix(X_test_emb_product * FINAL_EMBEDDING_SCALE)
])


# Final product-category classifier.
# LinearSVC works well with high-dimensional TF-IDF features.
product_clf = LinearSVC(
    C=FINAL_PRODUCT_C,
    class_weight="balanced",
    max_iter=5000,
    random_state=42
)


# Train the product model using the combined TF-IDF + MiniLM representation.
product_clf.fit(X_train_product_final, y_train_product)


# Predict product labels for validation and test data.
pred_valid_product = product_clf.predict(X_valid_product_final)
pred_test_product = product_clf.predict(X_test_product_final)

## 8. Validation evaluation

This evaluates the final selected system:
- hazard predictions from the TF-IDF hazard model
- product predictions from the TF-IDF + MiniLM product model


In [ ]:
# Evaluate the final model on the validation set.
f1_hazard, f1_product_conditional, final_official_score = official_score(
    y_valid_hazard,
    pred_valid_hazard,
    y_valid_product,
    pred_valid_product
)

print("FINAL MODEL")
print("Hazard model: TF-IDF word/char features + SGDClassifier")
print("Product model: TF-IDF word/char features + MiniLM embeddings + LinearSVC")
print("MiniLM embedding scale:", FINAL_EMBEDDING_SCALE)
print("Product LinearSVC C:", FINAL_PRODUCT_C)
print()
print("F1 hazard:", f1_hazard)
print("F1 product when hazard correct:", f1_product_conditional)
print("Official validation score:", final_official_score)

FINAL MODEL
Hazard model: TF-IDF word/char features + SGDClassifier
Product model: TF-IDF word/char features + MiniLM embeddings + LinearSVC
MiniLM embedding scale: 0.5
Product LinearSVC C: 2.0

F1 hazard: 0.8582462820456271
F1 product when hazard correct: 0.7688351441658579
Official validation score: 0.8135407131057425


## 9. Create Kaggle submission


In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "hazard-category": pred_test_hazard,
    "product-category": pred_test_product
})

# Check the submission size and missing values.
print("Submission shape:", submission.shape)
print("\nMissing values:")
print(submission.isnull().sum())

display(submission.head())

# Save the submission as a CSV file.
submission.to_csv("submission_tfidf_minilm_product.csv", index=False)
print("Saved file: submission_tfidf_minilm_product.csv")

Submission shape: (997, 3)

Missing values:
id                  0
hazard-category     0
product-category    0
dtype: int64


,id,hazard-category,product-category
0,0,biological,"meat, egg and dairy products"
1,1,biological,"meat, egg and dairy products"
2,2,biological,"meat, egg and dairy products"
3,3,biological,"meat, egg and dairy products"
4,4,foreign bodies,ices and desserts


Saved file: submission_tfidf_minilm_product.csv


In [ ]:
# Download the submission file to your local computer.

from google.colab import files
files.download("submission_tfidf_minilm_product.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>